## LAB 01 — Ingestão CRM (challenge 003)

Lê os CSVs de `dados/` e grava tabelas Delta. Colunas **somente as do CSV**. Não normalizar `GTXPro` na ingestão — o mismatch é parte do contrato.

### Controle de Alteração de Versões

| versão | data | autor | e-mail | alterações |
| --- | --- | --- | --- | --- |
| 1.2 | 21-SET-2026 | Adriano Aguiar | — | Troca para os CSVs CRM do challenge 003 |

### Contrato dos CSVs (contado no disco 2026-09-21)

| Arquivo | Linhas | Colunas |
| --- | ---: | --- |
| `accounts.csv` | 85 | `account`, `sector`, `year_established`, `revenue`, `employees`, `office_location`, `subsidiary_of` |
| `products.csv` | 7 | `product`, `series`, `sales_price` |
| `sales_teams.csv` | 35 | `sales_agent`, `manager`, `regional_office` |
| `sales_pipeline.csv` | 8800 | `opportunity_id`, `sales_agent`, `product`, `account`, `deal_stage`, `engage_date`, `close_date`, `close_value` |
| `metadata.csv` | 21 | `Table`, `Field`, `Description` — dicionário, não fato |

Joins por nome: `account`, `product`, `sales_agent`. Hub = `sales_pipeline`.

Armadilhas do dump (CSV ganha o brief):

- `GTXPro` (1480) **não** junta em `GTX Pro`. Normalizar só na análise, depois de contar o órfão.
- `account` null em 1425 deals abertos (Prospecting + Engaging). Scoring **não** pode exigir inner join.
- 5 agentes em `sales_teams` sem deal.
- `close_value`: Lost = 0; aberto = null; Won > 0.
- `sector` tem typo `technolgy`; `office_location` tem `Philipines`.
- `revenue` está em milhões USD (ver `metadata`).

### Como executar no Databricks

1. Importe a pasta `notebook-build-003-lead-scorer` para o Workspace (incluindo `dados/`).
2. Widgets: `catalog_name` (default `workspace`), `schema_name` (`crm`), `data_path` (vazio = auto).
3. `data_path` se o auto-detect falhar: `/Workspace/Users/<voce>/notebook-build-003-lead-scorer/dados`.
4. Cluster ou serverless → **Run All**.
5. Lab 02 e Lab 04 leem `workspace.crm.accounts` / `products` / `sales_teams` / `sales_pipeline`.

In [ ]:
from pathlib import Path

EXPECTED = {
    "accounts": {
        "rows": 85,
        "columns": [
            "account", "sector", "year_established", "revenue",
            "employees", "office_location", "subsidiary_of",
        ],
    },
    "products": {
        "rows": 7,
        "columns": ["product", "series", "sales_price"],
    },
    "sales_teams": {
        "rows": 35,
        "columns": ["sales_agent", "manager", "regional_office"],
    },
    "sales_pipeline": {
        "rows": 8800,
        "columns": [
            "opportunity_id", "sales_agent", "product", "account",
            "deal_stage", "engage_date", "close_date", "close_value",
        ],
    },
    "metadata": {
        "rows": 21,
        "columns": ["Table", "Field", "Description"],
    },
}

INT_COLUMNS = {"year_established", "employees", "sales_price"}
DOUBLE_COLUMNS = {"revenue", "close_value"}
DATE_COLUMNS = {"engage_date", "close_date"}


def widget_or_default(name: str, default: str) -> str:
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


def resolve_data_path(explicit: str) -> str:
    marker = "sales_pipeline.csv"
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    try:
        notebook_path = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        parts = [part for part in notebook_path.split("/") if part]
        if "notebook-build-003-lead-scorer" in parts:
            idx = parts.index("notebook-build-003-lead-scorer")
            repo = "/".join(parts[: idx + 1])
            candidates.append(Path("/Workspace") / repo / "dados")
            candidates.append(Path("/") / repo / "dados")
    except Exception:
        pass
    here = Path.cwd()
    candidates.extend(
        [
            here / "dados",
            here.parent / "dados",
            here.parent.parent / "dados",
        ]
    )
    for candidate in candidates:
        if (candidate / marker).exists():
            return str(candidate)
    raise FileNotFoundError(
        "Não achei sales_pipeline.csv. Defina o widget data_path "
        "para a pasta dados/ no Workspace ou em um Volume."
    )


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "crm")
data_path = resolve_data_path(widget_or_default("data_path", ""))
print(f"catalog={catalog_name} schema={schema_name}")
print(f"data_path={data_path}")

In [ ]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"USE SCHEMA {schema_name}")

In [ ]:
from pyspark.sql import functions as F

loaded = {}
for entity_name, contract in EXPECTED.items():
    csv_path = str(Path(data_path) / f"{entity_name}.csv")
    table_name = f"{catalog_name}.{schema_name}.{entity_name}"
    sdf = (
        spark.read.option("header", "true")
        .option("inferSchema", "true")
        .option("nullValue", "")
        .option("mode", "PERMISSIVE")
        .csv(csv_path)
    )
    missing = [col for col in contract["columns"] if col not in sdf.columns]
    extra = [col for col in sdf.columns if col not in contract["columns"]]
    if missing or extra:
        raise ValueError(
            f"{entity_name}: colunas fora do CSV. missing={missing} extra={extra}"
        )
    sdf = sdf.select(*contract["columns"])
    for column in sdf.columns:
        if column in INT_COLUMNS:
            sdf = sdf.withColumn(column, F.col(column).cast("int"))
        elif column in DOUBLE_COLUMNS:
            sdf = sdf.withColumn(column, F.col(column).cast("double"))
        elif column in DATE_COLUMNS:
            sdf = sdf.withColumn(column, F.to_date(F.col(column).cast("string")))
    row_count = sdf.count()
    if row_count != contract["rows"]:
        raise ValueError(
            f"{entity_name}: esperado {contract['rows']} linhas, lido {row_count}"
        )
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    loaded[entity_name] = {"rows": row_count, "cols": sdf.columns}
    print(f"{table_name} rows={row_count}")

loaded

In [ ]:
display(spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}"))

In [ ]:
for entity_name in EXPECTED:
    print(f"\n===== {entity_name} =====")
    display(spark.table(f"{catalog_name}.{schema_name}.{entity_name}").limit(5))